# 14 — Pattern viewer

Chart-pattern **shapes** and **S/R zones** on one price chart. Click a name in the legend to show or hide it.

Each pattern draws four things, all under its single legend entry:

| drawn | meaning |
|---|---|
| solid line + dots | the shape's **pivots**, in time order — the outline |
| dashed lines | the **upper and lower trendlines** through those pivots |
| amber line | the **neckline** that had to break for the pattern to count |
| dotted line | the **measured-move target** (pattern height projected from the break) |

**Nothing here is drawn early.** A shape appears only from the bar it *confirmed* — its pivots
were all confirmed beforehand and price had closed through the neckline. That is why a pattern
sometimes looks obvious in hindsight but is drawn later than you would have spotted it by eye:
the chart shows what was actually knowable at the time.

Only section 1 needs editing.

## 1. Config — edit here

In [1]:
# the kernel starts in notebooks/, so put the repo root on the path (same as the other notebooks)
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant

SYMBOL, TF   = "BTCUSDT", "1m"
START, END   = "2026-08-20", "2026-08-25"
SOURCE       = "binance"          # or "dukascopy" with SYMBOL="XAUUSD"

# what to draw
PATTERNS = [
    "symmetrical_triangle",
    "descending_triangle",
    "ascending_triangle",
    "head_shoulders",
]
SHOW_SR_ZONES = True

# which ones are ON when the chart opens (the rest start hidden; toggle from the legend).
# use "sr_zones" for the zones, or tuple(PATTERNS) to start with everything on.
START_VISIBLE = ("sr_zones", "symmetrical_triangle")

# --- how strict, overall -------------------------------------------------------------
# "major"  = only big, well-touched structures      "normal" = major + intermediate (default)
# "minor"  = everything, including small shapes
SENSITIVITY = "normal"

# --- per-pattern overrides -----------------------------------------------------------
# Anything you leave out just uses the SENSITIVITY preset. Every field of PatternSpec:
#   min_touches_upper / min_touches_lower  pivots each trendline must pass through (3 = a real line;
#                                          2 is meaningless because two points always fit exactly)
#   min_span_bars    how wide the shape must be, in bars (absolute - a triangle is a triangle
#                    whether you loaded two days or two years)
#   span_cap_frac    never demand more than this fraction of the loaded chart
#   min_height_atr   reject shapes shorter than this many ATR
#   line_tol_atr     how far off the line a pivot may sit and still count as a touch
#   flat_tol_atr     a line is "flat" if it moves less than this many ATR across the whole span
#   level_tol        equal-highs/lows tolerance (double tops, H&S shoulder symmetry)
PATTERN_SETTINGS = {
    # "ascending_triangle":   dict(min_touches_upper=4, min_span_bars=150),
    # "descending_triangle":  dict(min_touches_lower=4),
    # "symmetrical_triangle": dict(min_span_bars=200),
    # "head_shoulders":       dict(level_tol=0.15),
}

# --- detector plumbing (rarely needs touching) ---------------------------------------
ATR_MULT        = 3.5    # zigzag swing size: LOWER = more, smaller pivots
LOOKBACK_PIVOTS = 24     # how many recent pivots one shape may span
MAX_WAIT        = 150    # bars a completed shape waits for its break
OVERLAP_FRAC    = 0.5    # two same-name shapes overlapping more than this -> keep the better one
EXTEND          = 12     # bars to extend the lines past the break

# --- S/R zone sensitivity -------------------------------------------------------------
ZONE_LR      = 10        # pivot look-around each side
CLUSTER_PCT  = 0.15      # merge levels within this % of each other
MIN_TOUCHES  = 2         # hide zones touched fewer times than this

HEIGHT = 820

DETECT = dict(sensitivity=SENSITIVITY, settings=PATTERN_SETTINGS, atr_mult=ATR_MULT,
              lookback_pivots=LOOKBACK_PIVOTS, max_wait=MAX_WAIT, overlap_frac=OVERLAP_FRAC)

## 2. Data

In [2]:
from quant.data import get_ohlcv

df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)
print(f"{len(df):,} bars   {df.t.min()}  ->  {df.t.max()}")

[07:08:25] INFO | quant.data | cache EXTEND binance BTCUSDT 1m -> incremental fetch
[07:08:25] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=1m, start=2026-08-20 00:00:00+00:00, end=2026-08-25 00:00:00+00:00
[07:08:25] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[07:08:25] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSDT_1m.parquet
[07:08:25] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\.partials\binance_spot_BTCUSDT_1m
[07:08:25] INFO | 05. Cache: HIT | rows=1,054,088 (main_rows=1,054,088, partial_files=0) | range=2024-08-22 00:00:00+00:00 -> 2026-08-24 00:07:00+00:00
[07:08:25] INFO | 06. Fetch plan 1/1: 2026-08-24 00:08:00+00:00 -> 2026-08-25 00:00:00+00:00
[07:08:25] INFO | 07. Starting fetch range 1/1.


BTCUSDT 1m:   0%|          | 0/2 [00:00<?, ?page/s]

[07:08:30] INFO | 08. Completed fetch range 1/1 | rows=1,433 | total_rows_fetched=1,433 | requests=2 | retries=0 | elapsed=5.6s
[07:08:32] INFO | 09. Cache consolidated and saved (parquet) | rows=1,055,521 | range=2024-08-22 00:00:00+00:00 -> 2026-08-25 00:00:00+00:00 | cleared_partial_files=2
[07:08:32] INFO | 10. Fetch summary | new_rows=1,433 | successful_pages=2 | requests=2 | retries=0 | partial_saves=2 | elapsed=6.9s
[07:08:32] INFO | 11. Returning slice | rows=7,201 | range=2026-08-20 00:00:00+00:00 -> 2026-08-25 00:00:00+00:00


7,201 bars   2026-08-20 00:00:00+00:00  ->  2026-08-25 00:00:00+00:00


## 3. What was found

A quick count before plotting — if a pattern shows 0 here, lower `ATR_MULT` or widen the date range.

In [3]:
from quant.indicators import chart_pattern_shapes

shapes = chart_pattern_shapes(df, patterns=PATTERNS, **DETECT)
if len(shapes):
    print(shapes.pattern.value_counts().to_string())
    print()
    med, mx = shapes.span_bars.median(), shapes.span_bars.max()
    print(f"median width {med:.0f} bars ({100*med/len(df):.1f}% of the chart), widest {mx:.0f}")
else:
    print("no patterns found - try SENSITIVITY='minor' or a lower ATR_MULT")
shapes[["t", "pattern", "direction", "span_bars", "touches_upper", "touches_lower",
        "entry", "stop", "target"]].tail(10)

pattern
symmetrical_triangle    16
head_shoulders          10
descending_triangle      2

median width 141 bars (2.0% of the chart), widest 464


,t,pattern,direction,span_bars,touches_upper,touches_lower,entry,stop,target
18,2026-08-23 17:31:00+00:00,symmetrical_triangle,-1,228,4,4,77228.879729,77640.58,76390.299729
19,2026-08-23 18:05:00+00:00,descending_triangle,-1,107,4,3,77163.915307,77475.00,76832.915307
20,2026-08-23 18:07:00+00:00,head_shoulders,-1,83,0,0,77146.100000,77475.00,76817.200000
21,2026-08-24 00:43:00+00:00,symmetrical_triangle,-1,326,4,3,77291.781083,78052.85,76438.961083
22,2026-08-24 03:36:00+00:00,head_shoulders,-1,120,0,0,77013.230000,77717.28,76309.180000
23,2026-08-24 04:48:00+00:00,symmetrical_triangle,-1,241,4,3,76979.239298,77793.85,76068.999298
24,2026-08-24 08:15:00+00:00,head_shoulders,-1,112,0,0,77219.000000,77790.19,76647.810000
25,2026-08-24 16:54:00+00:00,head_shoulders,-1,96,0,0,79052.140000,80000.00,78104.280000
26,2026-08-24 19:19:00+00:00,symmetrical_triangle,-1,131,5,3,78832.940855,79198.00,78103.550855
27,2026-08-24 21:54:00+00:00,symmetrical_triangle,-1,163,3,3,78982.720269,79118.42,78424.300269


## 4. Chart — toggle each pattern from the legend

In [4]:
from quant.viz import pattern_figure

fig = pattern_figure(
    df, patterns=PATTERNS, start_visible=START_VISIBLE,
    sr_zones=SHOW_SR_ZONES, zones_left=ZONE_LR, zones_right=ZONE_LR,
    cluster_pct=CLUSTER_PCT, min_touches=MIN_TOUCHES,
    extend=EXTEND, detect_kwargs=DETECT,
    title=f"{SYMBOL} {TF}  {START} to {END}", height=HEIGHT)
# fig

### Open it in a browser (bigger, and it survives the notebook being closed)

In [6]:
from quant.viz import open_in_browser

open_in_browser(fig)

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/chart.html')

---
### Zoom to one pattern

Set `PICK` to a row number from section 3 to focus the chart around that occurrence.

In [6]:
# PICK = -1        # -1 = the most recent one
# PAD  = 60        # bars of context each side

# if len(shapes):
#     row = shapes.iloc[PICK]
#     lo = max(int(row.start_i) - PAD, 0)
#     hi = min(int(row.confirm_i) + PAD, len(df) - 1)
#     sub = df.iloc[lo:hi + 1].reset_index(drop=True)
#     zoom = pattern_figure(sub, patterns=PATTERNS, start_visible=tuple(PATTERNS) + ("sr_zones",),
#                           sr_zones=SHOW_SR_ZONES, zones_left=ZONE_LR, zones_right=ZONE_LR,
#                           cluster_pct=CLUSTER_PCT, min_touches=MIN_TOUCHES,
#                           extend=EXTEND, detect_kwargs=DETECT, height=700,
#                           title=f"{row.pattern} confirmed {row.t}")
#     zoom.show()
# else:
#     print("nothing to zoom to")

---
**A note on what you are looking at.** Each trendline is FITTED through 3+ swing pivots and
extended across the shape — a line through only two points carries no information, because two
points always fit exactly. Shapes appear from the bar they CONFIRMED (all pivots already
confirmed, price closed through the boundary), so they show up later than you would spot them
by eye; that is what was actually knowable at the time.

The research in `docs/WHAT_WORKS.md` found chart patterns do not predict direction. What they
do predict is size — read them as "expect a bigger move", not "expect this direction".
